In [1]:
# type: ignore


import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI


load_dotenv()

llm_model = ChatOpenAI(
    model=os.environ.get("MODEL_NAME"),
    temperature=0,
    base_url=os.environ.get("COMPATIBLE_BASE_URL"),
    api_key=os.environ.get("COMPATIBLE_API_KEY"),
    extra_body={"enable_thinking": False},
)

In [2]:
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field


class GradeAnswer(BaseModel):
    """Binary score indicating whether a generated answer adequately addresses the user's question."""

    binary_score: bool = Field(
        description="Answer addresses the question, 'yes' or 'no'"
    )


structured_llm_grader = llm_model.with_structured_output(GradeAnswer)

system_prompt = """You are a grader assessing whether an answer addresses / resolves a question \n 
     Give a binary score 'yes' or 'no'. Yes' means that the answer resolves the question.
     
     You MUST output a valid JSON object with ONLY the key "binary_score"."""

answer_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "User question: \n\n {question} \n\n LLM generation: {generation}"),
    ]
)

answer_grader = answer_prompt | structured_llm_grader

In [3]:
answer_grader.invoke(
    {
        "question": "什么是 Agentic RAG?",
        "generation": "Agentic RAG 融合自主智能体与 RAG，支持动态推理、多步检索与自我优化。",
    }
)

GradeAnswer(binary_score=True)

In [4]:
answer_grader.invoke(
    {
        "question": "什么是 Agentic RAG?",
        "generation": "RAG是一种结合外部知识检索与 LLM 生成能力的技术，用于生成更准确、事实性强且基于最新信息的回答。",
    }
)

GradeAnswer(binary_score=False)